# Notebook 11: Personalización de Data Docs

**Duración**: 40 minutos | **Nivel**: Avanzado

## Introducción

Aprende a personalizar los Data Docs para tu organización.

### Objetivos:
1. Agregar metadata personalizada
2. Organizar expectativas por categorías
3. Mejorar la presentación visual
4. Crear reportes ejecutivos

In [ ]:
import great_expectations as gx
import pandas as pd

df = pd.read_csv("../data/ventas_sucias.csv")

context = gx.get_context(mode="ephemeral")
datasource = context.data_sources.add_pandas(name="ventas_ds")
asset = datasource.add_dataframe_asset(name="ventas")
batch_def = asset.add_batch_definition_whole_dataframe("batch_completo")

## Metadata Enriquecida

In [ ]:
suite = context.suites.add(gx.ExpectationSuite(name="suite_documentada"))

# Agregar metadata rica
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="customer_id",
        meta={
            "dimension": "Completitud",
            "criticidad": "Alta",
            "owner": "Equipo de Ventas",
            "descripcion": "Customer ID es obligatorio para procesar pedidos",
            "impacto_negocio": "Sin customer_id no podemos enviar el pedido",
            "accion_falla": "Rechazar registro y notificar al sistema origen"
        }
    )
)

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="price",
        min_value=0.01,
        max_value=10000,
        meta={
            "dimension": "Validez",
            "criticidad": "Alta",
            "owner": "Equipo de Finanzas",
            "descripcion": "Precio debe estar en rango razonable",
            "regla_negocio": "Productos entre $0.01 y $10,000",
            "accion_falla": "Marcar para revisión manual"
        }
    )
)

suite.save()
print(" Suite con metadata enriquecida creada")

## Organización por Categorías

In [ ]:
# Crear múltiples suites organizadas
suite_critica = context.suites.add(gx.ExpectationSuite(name="validaciones_criticas"))
suite_advertencia = context.suites.add(gx.ExpectationSuite(name="validaciones_advertencia"))

# Críticas
suite_critica.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="order_id")
)

# Advertencias
suite_advertencia.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="product_category",
        mostly=0.95
    )
)

suite_critica.save()
suite_advertencia.save()

print(" Suites organizadas por severidad")

## Generar Data Docs Personalizados

In [ ]:
# Validar todas las suites
for suite_name in ["suite_documentada", "validaciones_criticas", "validaciones_advertencia"]:
    suite = context.suites.get(suite_name)
    val_def = context.validation_definitions.add(
        gx.ValidationDefinition(
            data=batch_def,
            suite=suite,
            name=f"val_{suite_name}"
        )
    )
    resultado = val_def.run(batch_parameters={"dataframe": df})
    print(f"{suite_name}: {'' if resultado.success else ''}")

# Generar Data Docs
context.build_data_docs()
print("\n Data Docs personalizados generados")
context.open_data_docs()

##  Resumen

1.  Metadata enriquece documentación
2.  Organización por severidad mejora claridad
3.  Data Docs son personalizables
